# The REG101_APPLN table

Welcome to the first table of PATSTAT Register, namely table **REG101_APPLN**. This table contains:
* EP applications with their identifiers and some additional data. In case of a Euro-PCT application, i.e. an international application which has entered the EP regional phase, it is assigned the same number of the international application. If no international application number is given, then it is an EP direct filing.
* International applications, that have not (yet) entered the EP regional phase. 

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG101_APPLN
from sqlalchemy import func
import pandas as pd

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## ID (Primary Key)

A technical identifier for an application, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [2]:
i = db.query(
    REG101_APPLN.id
).limit(1000)

df = patstat.df(i)
df

,id
0,12156108
1,11808693
2,8016148
3,12850521
4,11837267
...,...
995,15770487
996,19916601
997,23714514
998,95307555


## APPLN_ID

Application identifier. `APPLN_ID` > 0 indicates that the application is either an EP direct application or a PCT international application which has entered the EP regional phase. The corresponding EP application in the PATSTAT Global database is linkable via the attribute `APPLN_ID` which occurs in both databases. `APPLN_ID` = 0 indicates that this application has not entered the EP regional phase. Therefore, the PATSTAT Global database does not contain a corresponding EP application.

In [3]:
i = db.query(
    REG101_APPLN.id,
    REG101_APPLN.appln_id
).limit(1000)

df = patstat.df(i)
df

,id,appln_id
0,85110061,16587946
1,9736309,0
2,2780282,0
3,20959669,0
4,18778636,0
...,...,...
995,14740155,420416339
996,99913271,17423037
997,11182929,337190841
998,17704071,475206571


## APPLN_AUTH

Office where the application has been filed. In table `REG101_APPLN` this attribute is always ‘EP’, because the scope of this database is European patents.

In [4]:
# Use the count function in the query and rename the column via the label command
group_q = db.query(
    func.count(REG101_APPLN.appln_id).label('total_applications'),
    REG101_APPLN.appln_auth
).group_by(
    REG101_APPLN.appln_auth  # Here we use the group_by function on the 'appln_auth' field
).order_by(
    func.count(REG101_APPLN.appln_id)
)

# Convert it in a dataframe
grouped_res = patstat.df(group_q)
grouped_res

,total_applications,appln_auth
0,7082444,EP


The only application authority is indeed 'EP', since PATSTAT Register concerns EPO applications and patents only.

## APPLN_NR

Application number as issued by the application authority. Note that the attribute must not be a numerical attribute but a text string attribute because leading zeros are significant.

In [5]:
appln_nr = db.query(
    REG101_APPLN.appln_nr,
    REG101_APPLN.appln_id
).limit(1000)

appln_nr_df = patstat.df(appln_nr)
appln_nr_df

,appln_nr,appln_id
0,82810418,16509034
1,04803104,16175362
2,18167333,493116849
3,94910379,17074380
4,19760615,518848372
...,...,...
995,23775399,0
996,21204094,559363923
997,04300486,16126139
998,03794501,16079682


Except for the value 0, the `appln_id` attribute is associated to only one `appln_nr`.

In [6]:
tot_nr = db.query(
    func.count(REG101_APPLN.appln_nr).label('tot_appln_nr'),
    REG101_APPLN.appln_id
).group_by(
    REG101_APPLN.appln_id
).order_by(
    func.count(REG101_APPLN.appln_nr).label('tot_appln_nr')
)

tot_nr_df = patstat.df(tot_nr)
tot_nr_df

,tot_appln_nr,appln_id
0,1,341684325
1,1,341078239
2,1,357783
3,1,406635061
4,1,553864925
...,...,...
4470052,1,437009
4470053,1,585072067
4470054,1,335804496
4470055,1,16797748


## APPLN_FILING_DATE

Date on which the application was received at the Patent Authority.

Notice that there are several applications with filing application year equal to 9&nbsp;999. These correspond to missing dates.

In [7]:
missing_dates = db.query(
    REG101_APPLN.appln_filing_date,
    REG101_APPLN.appln_id
).order_by(
    REG101_APPLN.appln_filing_date.desc()
).limit(50000)

missing_dates_df = patstat.df(missing_dates)
missing_dates_df

,appln_filing_date,appln_id
0,9999-12-31,0
1,9999-12-31,0
2,9999-12-31,0
3,9999-12-31,0
4,9999-12-31,0
...,...,...
49995,2024-05-09,0
49996,2024-05-09,0
49997,2024-05-09,0
49998,2024-05-09,0


Missing dates may correspond to withdrawn applications. Whatever the reason, these cases represent a small portion of the database.

In [8]:
num_missing_dates = db.query(
    REG101_APPLN.appln_filing_date,
    func.count(REG101_APPLN.appln_id).label('num_appln')
).filter(
    REG101_APPLN.appln_filing_date == '9999-12-31'
).group_by(
    REG101_APPLN.appln_filing_date
).order_by(
    REG101_APPLN.appln_filing_date.desc()
)

num_missing_dates_df = patstat.df(num_missing_dates)
num_missing_dates_df

,appln_filing_date,num_appln
0,9999-12-31,22


## FILING_LG

The language in which the application was filed.

Let's see which languages are the most frequent in the database.

In [9]:
lg = db.query(
    REG101_APPLN.filing_lg,
    func.count(REG101_APPLN.appln_id).label('tot_appln')
).group_by(
    REG101_APPLN.filing_lg
).order_by(
    func.count(REG101_APPLN.appln_id).desc()
).limit(20)

lg_df = patstat.df(lg)
lg_df

,filing_lg,tot_appln
0,en,3988956
1,de,934053
2,ja,844059
3,zh,574607
4,fr,298698
5,ko,203735
6,it,81591
7,es,38776
8,ru,22675
9,nl,20858


## STATUS

Status of the application. The status of granted patents (e. g. still valid or not) is not included here. The domain consists of numbers from 1 to 18. This attribute permits to link this table to table REG403_APPLN_STATUS. Therefore, its meaning and utility will be explained later on in this series of notebooks.

In [10]:
status = db.query(
    REG101_APPLN.status
).limit(1000)

status_df = patstat.df(status)
status_df

,status
0,7
1,7
2,10
3,7
4,14
...,...
995,17
996,14
997,7
998,10


Raw numbers like this are not meaningful. We can see the actual text corresponding to the status joining table REG101 with table REG403 as said above.

In [11]:
from epo.tipdata.patstat.database.models import REG403_APPLN_STATUS

text_status = db.query(
    REG101_APPLN.status,
    REG403_APPLN_STATUS.status_text
).distinct(
    REG101_APPLN.status
).join(
    REG101_APPLN, REG403_APPLN_STATUS.status == REG101_APPLN.status
).order_by(
    REG101_APPLN.status
)

text_status_df = patstat.df(text_status)
text_status_df

,status,status_text
0,1,Patent revoked by proprietor
1,2,The patent has been limited
2,3,Patent maintained as amended
3,4,Patent revoked
4,5,Opposition rejected
5,6,Opposition procedure closed
6,7,No opposition filed within time limit
7,8,The patent has been granted
8,9,The application has been withdrawn
9,10,The application is deemed to be withdrawn


## INTERNAT_APPLN_ID

Application ID of the international application. This attribute allows for each PCT application in PATSTAT EP Register to easily retrieve the corresponding PCT application in PATSTAT Global. You just have to join the non-zero values of attribute `REG101_APPLN.INTERNAT_APPLN_ID` with the attribute `TLS201_APPLN.APPLN_ID`.

In [12]:
from epo.tipdata.patstat.database.models import TLS201_APPLN

internat = db.query(
    REG101_APPLN.internat_appln_id,
    TLS201_APPLN.appln_id
).filter(
    REG101_APPLN.internat_appln_id != 0
).join(
    REG101_APPLN, TLS201_APPLN.appln_id == REG101_APPLN.internat_appln_id
)

internat_df = patstat.df(internat)
internat_df

,internat_appln_id,appln_id
0,621835997,621835997
1,566925561,566925561
2,450078654,450078654
3,562535561,562535561
4,449604881,449604881
...,...,...
5011316,329289121,329289121
5011317,558609014,558609014
5011318,539231277,539231277
5011319,518942967,518942967


## INTERNAT_APPLN_NR

International application number. It consists of up to fifteen characters.

In [13]:
internat_appln_nr = db.query(
    REG101_APPLN.internat_appln_id,
    REG101_APPLN.internat_appln_nr
).limit(1000)

internat_appln_nr_df = patstat.df(internat_appln_nr)
internat_appln_nr_df

,internat_appln_id,internat_appln_nr
0,0,
1,17472453,WO2004EP12516
2,0,
3,15666677,WO1994EP00675
4,518943028,WO2019IB51643
...,...,...
995,599389828,WO2023SG50176
996,0,
997,0,
998,45712135,WO2003US26508


## BIO_DEPOSIT

An indicator telling whether one or more deposits of biological material have been made or not. The domain consists of one ASCII character: Y or N.

Let's count how many Y and N there are in the database.

In [14]:
bio = db.query(
    REG101_APPLN.bio_deposit,
    func.count(REG101_APPLN.appln_id).label('number_of_applications')
).group_by(
    REG101_APPLN.bio_deposit
).order_by(
    func.count(REG101_APPLN.appln_id).desc()
)

bio_df = patstat.df(bio)
bio_df

,bio_deposit,number_of_applications
0,N,7060531
1,Y,21913
